In [1]:
import os
import json
import time
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

TOOL_FILE_DIR = Path("./tool_file")
TOOL_FILE_DIR.mkdir(exist_ok=True)

In [2]:
def dump(tool_name: str, data, run_id: str) -> str:
    """모든 툴의 결과는 이 함수를 통해서만 파일로 떨군다. 반환값은 파일 경로 하나.
    에이전트는 이 경로를 읽어서 검증하고, 다른 툴에 넘길 때도 데이터가 아니라 이 경로를 넘긴다."""
    path = TOOL_FILE_DIR / f"{run_id}_{tool_name}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(path)

In [3]:
def _kakao_keyword(query: str, page: int = 1, category_code: str = "") -> tuple[list[dict], bool]:
    """카카오 키워드 검색 1페이지. (결과 리스트, is_end) 반환."""
    params = {"query": query, "size": 15, "page": page}
    if category_code:
        params["category_group_code"] = category_code

    resp = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params,
        timeout=5,
    )
    resp.raise_for_status()
    body = resp.json()

    places = [
        {
            "id": d["id"],
            "name": d["place_name"],
            "category": d["category_name"],
            "address": d["road_address_name"] or d["address_name"],
            "url": d["place_url"],
            "source": "kakao_local",
        }
        for d in body["documents"]
    ]
    return places, body["meta"]["is_end"]

In [4]:
_kakao_keyword("창원 귀산 카페")

([{'id': '1181055498',
   'name': '엘리브',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 귀산로 19',
   'url': 'http://place.map.kakao.com/1181055498',
   'source': 'kakao_local'},
  {'id': '1126054801',
   'name': '더로드101 귀산',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로486번길 61-8',
   'url': 'http://place.map.kakao.com/1126054801',
   'source': 'kakao_local'},
  {'id': '1570418656',
   'name': '카페모조',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로486번길 60',
   'url': 'http://place.map.kakao.com/1570418656',
   'source': 'kakao_local'},
  {'id': '1114671449',
   'name': '하우요커피',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로 524-6',
   'url': 'http://place.map.kakao.com/1114671449',
   'source': 'kakao_local'},
  {'id': '1719931187',
   'name': '로스터리퐁',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로486번길 69',
   'url': 'http://place.map.kakao.com/1719931187',
   'source': 'kakao_local'},
  {'id': '1846671408',
   'name': '갱고반지하',
   'categor

In [5]:
import re
import html

def _strip_tags(text: str) -> str:
    """네이버 응답의 <b> 하이라이트 태그와 HTML 엔티티를 제거한다."""
    return html.unescape(re.sub(r"<[^>]+>", "", text)).strip()


def _naver_blog(query: str, display: int = 30, sort: str = "sim") -> list[dict]:
    """네이버 블로그 검색 1회 호출.

    display: 1~100 (기본 10)
    sort: 'sim'(정확도순) 또는 'date'(최신순)
    """
    resp = requests.get(
        "https://openapi.naver.com/v1/search/blog",
        headers={
            "X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
            "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"],
        },
        params={"query": query, "display": display, "sort": sort},
        timeout=5,
    )
    resp.raise_for_status()

    return [
        {
            "title": _strip_tags(item["title"]),
            "description": _strip_tags(item["description"]),
            "link": item["link"],
            "postdate": item.get("postdate", ""),
        }
        for item in resp.json()["items"]
    ]

In [6]:
_naver_blog("창원 가포 오션뷰 카페")

[{'title': '마산 창원 대형 해외 st 오션뷰 카페 : 가포 제주도 감성 맛집 추천! ....',
  'description': '한시간 반에서 2시간 정도 소요됩니다 :) 카페 지중해 : 경상남도 창원시 마산합포구 가포로 501-201... 바다뷰카페, 마산오션뷰, 창원오션뷰, 경남오션뷰카페, 대형카페, 마산대형카페, 창원대형카페, 베이커리카페...',
  'link': 'https://blog.naver.com/ng712/224366715437',
  'postdate': '20260803'},
 {'title': '마산 가포 오션뷰 카페 페리데스, 커피보다 분위기에 먼저 반했던 ....',
  'description': '예쁜 곳보다 오래 머물고 싶은 공간을 찾는다면 페리데스를 추천드립니다. #마산카페 #가포카페 #페리데스 #마산오션뷰카페 #마산베이커리카페 #창원카페추천 #가포드라이브 #마산데이트 #경남카페 #대형카페',
  'link': 'https://blog.naver.com/dwd0684/224336449681',
  'postdate': '20260704'},
 {'title': '[창원카페추천] 페리데스 반전매력 오션뷰 베이커리 맛집 내돈내산 ....',
  'description': '[창원카페추천] 페리데스 오션뷰 베이커리 맛집 내돈내산 후기 안녕하세요... 다녀와보니, 창원에서 뷰 좋은 베이커리 카페 찾으신다면 페리데스 진짜 추천드려요  가포 앞바다 보면서 갓 구운 빵이랑 커피 한 잔...',
  'link': 'https://blog.naver.com/canada179/224369721321',
  'postdate': '20260806'},
 {'title': '(창원) 마산 가포 오션뷰카페 지중해 후기',
  'description': 'ㅋㅋ 주차장도 넓어서 좋고 진짜 바다 바로 코앞인데다가 제주도 느낌까지 느낄 수 있는 카페라서 너무 좋았어요 가족 나들이, 데이트코스로도 너무 좋을것같아요 창원,마산 가포 오션뷰카페 지중해 

In [7]:
from google import genai
from google.genai import types

_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
UTIL_MODEL = "gemini-3.1-flash-lite"

def _llm(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 단발 호출. stateless, 텍스트만 반환."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"

    resp = _client.models.generate_content(
        model=UTIL_MODEL,
        contents=prompt,
        config=config,
    )
    return resp.text

In [8]:
import json

_EXTRACT_PLACE_NAMES_SYSTEM_PROMPT = """\
아래는 네이버 블로그 검색 결과다. 각 줄은 '제목 | 요약' 형식이다.
여기서 실제 존재하는 가게/장소의 고유 상호명만 뽑아라.
- 제목에 상호명이 있을 확률이 높다. 요약은 보조로만 참고한다.
- '카페 추천', '맛집 베스트10', '데이트코스' 같은 일반명사구는 상호명이 아니다. 
- 같은 매장이 여러 줄에 걸쳐 나오면 한 번만 적어라.
- 확실하지 않으면 넣지 마라.
- 결과는 문자열 배열 JSON으로만 출력한다. 예: ["블루보틀 성수", "어니언 성수"]
"""

def _extract_place_names(blog_items: list[dict]) -> list[str]:
    """네이버 블로그 검색 결과 뭉치에서 실제 매장/장소 상호명만 추출한다.
    title에 상호명이 있을 확률이 높고 description은 보조 참고용이다.
    반환값은 검증 전 후보 목록이며, 호출부에서 카카오 역조회로 실존을 확인해야 한다."""
    if not blog_items:
        return []

    lines = "\n".join(
        f"{i+1}. {it['title']} | {it['description']}"
        for i, it in enumerate(blog_items)
    )
    prompt = f"{_EXTRACT_PLACE_NAMES_SYSTEM_PROMPT}\n{lines}"

    raw = _llm(prompt, json_mode=True)
    try:
        names = json.loads(raw)
    except json.JSONDecodeError:
        return []

    if not isinstance(names, list):
        return []
    return [n.strip() for n in names if isinstance(n, str) and n.strip()]

In [9]:
from google.adk.tools import ToolContext


def collect_places(
    query: str,
    tool_context: ToolContext,
    target_count: int = 20,
    category_code: str = "",
) -> dict:
    """지역 키워드로 매장 후보 목록을 수집한다.

    카카오 로컬 검색을 우선 사용하며, 결과가 부족하면 재시도를 요청한다.
    3회까지 부족하면 네이버 블로그에서 매장명을 추출해 카카오로 검증한 뒤 보충한다.

    검색어는 '지역 + 짧은 업종어' 형태여야 한다. 서술형 문장은 결과가 거의 나오지 않는다.
      좋은 예: '성수동 카페', '성수동 카공카페'
      나쁜 예: '성수동에서 조용히 작업하기 좋은 카페'

    Args:
        query: 검색어. '지역 + 짧은 업종어' 조합.
        target_count: 확보하려는 결과 개수. 최대 45까지만 가능하다.
        category_code: 카테고리 필터. 'CE7'=카페, 'FD6'=음식점, 'AT4'=관광명소, 'CT1'=문화시설.
                       불필요하면 빈 문자열로 둔다.

    Returns:
        status가 'retry'면 결과가 부족한 것이므로, message의 안내에 따라
        query를 더 짧고 단순하게 바꿔서 이 툴을 다시 호출해야 한다.
        status가 'ok'면 수집이 끝난 것이므로 다시 호출하지 말고 file 경로를 사용한다.
        예시(재시도): {'status': 'retry', 'message': '결과가 부족합니다...'}
        예시(완료): {'status': 'ok', 'count': 20, 'file': './tool_file/collect_places_123.json'}
    """
    attempt_key = "temp:collect_attempts"
    attempts = tool_context.state.get(attempt_key, 0) + 1
    tool_context.state[attempt_key] = attempts

    places: dict[str, dict] = {}

    # 1) 카카오 로컬 페이지네이션
    page = 1
    while len(places) < target_count and page <= 45:
        batch, is_end = _kakao_keyword(query, page=page, category_code=category_code)
        for p in batch:
            places.setdefault(p["id"], p)
        if is_end:
            break
        page += 1

    # 2) 부족한데 아직 재시도 여유가 있으면 → 에이전트에게 쿼리 재구성 요청
    if len(places) < target_count * 0.5 and attempts < 3:
        return {
            "status": "retry",
            "message": (
                f"'{query}' 검색 결과가 {len(places)}개로 부족합니다. ({attempts}/3번째 시도) "
                "핵심 지역명은 유지하고 수식어를 제거해 더 짧은 업종어로 바꿔 다시 호출하세요. "
                "예: '조용히 작업하기 좋은 카페' → '카공카페' 또는 '스터디카페'"
            ),
        }

    # 3) 3회까지 부족하면 → 네이버 블로그에서 매장명 추출 후 카카오로 검증
    if len(places) < target_count * 0.5:
        blog_items = _naver_blog(query, display=30, sort="sim")
        for name in _extract_place_names(blog_items):
            if len(places) >= target_count:
                break
            verified, _ = _kakao_keyword(name, page=1)
            for p in verified:
                if p["id"] not in places:
                    p["source"] = "blog_derived"
                    places[p["id"]] = p
                    break

    result = list(places.values())[:target_count]
    return {
        "status": "ok",
        "count": len(result),
        "source_breakdown": {
            "local": sum(1 for p in result if p.get("source") == "kakao_local"),
            "blog_derived": sum(1 for p in result if p.get("source") == "blog_derived"),
        },
        "file": dump("collect_places", result, _run_id(tool_context)),
    }

In [10]:
def _run_id(tool_context: ToolContext) -> str:
    """현재 invocation의 식별자. 없으면 최초 호출 시점으로 생성한다."""
    key = "temp:run_id"
    rid = tool_context.state.get(key)
    if rid is None:
        rid = str(int(time.time() * 1000))
        tool_context.state[key] = rid
    return rid

In [11]:
class FakeToolContext:
    def __init__(self, invocation_id="test-run"):
        self.state = {}
        self.invocation_id = invocation_id

ctx = FakeToolContext()


ctx = FakeToolContext()
r1 = collect_places("창원 가포 오션뷰 카페 맛집", ctx)  # attempts=1
print(r1)
r2 = collect_places("창원 가포 오션뷰 카페 맛집", ctx)          # attempts=2
print(r2)
r3 = collect_places("창원 가포 오션뷰 카페 맛집", ctx)               # attempts=3 → 네이버 폴백
print(r3)

{'status': 'retry', 'message': "'창원 가포 오션뷰 카페 맛집' 검색 결과가 0개로 부족합니다. (1/3번째 시도) 핵심 지역명은 유지하고 수식어를 제거해 더 짧은 업종어로 바꿔 다시 호출하세요. 예: '조용히 작업하기 좋은 카페' → '카공카페' 또는 '스터디카페'"}
{'status': 'retry', 'message': "'창원 가포 오션뷰 카페 맛집' 검색 결과가 0개로 부족합니다. (2/3번째 시도) 핵심 지역명은 유지하고 수식어를 제거해 더 짧은 업종어로 바꿔 다시 호출하세요. 예: '조용히 작업하기 좋은 카페' → '카공카페' 또는 '스터디카페'"}
{'status': 'ok', 'count': 8, 'source_breakdown': {'local': 0, 'blog_derived': 8}, 'file': 'tool_file\\1786435143454_collect_places.json'}


In [12]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

APP_NAME = "wh_poc"
USER_ID = "user1"
SESSION_ID = "s1"

INSTRUCTION = """당신은 지역 장소를 찾아주는 어시스턴트입니다.

사용자가 장소를 찾아달라고 하면:
1. 사용자의 자연어 요청에서 '지역 + 짧은 업종어' 형태의 검색어를 만드세요.
   예: "성수동에서 조용히 작업할 카페" → "성수동 카공카페"
2. collect_places 툴을 호출하세요.
3. 반환된 status를 반드시 확인하세요.

status가 'retry'인 경우:
- message의 안내대로 query를 더 짧고 단순하게 바꿔서 collect_places를 즉시 다시 호출하세요.
- 사용자에게 되묻지 말고, 스스로 검색어를 바꿔서 재시도하세요.

status가 'ok'인 경우:
- 더 이상 collect_places를 호출하지 마세요.
- count와 file 경로를 사용자에게 알려주고 결과를 정리해 답변하세요.
"""

agent = LlmAgent(
    model="gemini-3.1-flash-lite",
    name="place_finder",
    description="지역 기반 장소 추천 에이전트",
    instruction=INSTRUCTION,
    tools=[collect_places],
)

runner = InMemoryRunner(agent=agent, app_name=APP_NAME)

In [13]:
async def run(query: str):
    await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    content = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f"[TOOL CALL] {part.function_call.name}({dict(part.function_call.args)})")
                elif part.function_response:
                    print(f"[TOOL RESULT] {part.function_response.response}")
                elif part.text and part.text.strip():
                    print(f"[TEXT] {part.text.strip()}")

await run("창원 가포 오션뷰 카페 맛집좀 찾아줘 어머니 모시고 갈꺼임")

d:\python_project\welcom_to_hometown\.venv\Lib\site-packages\google\adk\tools\function_tool.py:96: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


[TOOL CALL] collect_places({'category_code': 'CE7', 'target_count': 10, 'query': '창원 가포 카페'})
[TOOL RESULT] {'status': 'ok', 'count': 10, 'source_breakdown': {'local': 10, 'blog_derived': 0}, 'file': 'tool_file\\1786435145109_collect_places.json'}
[TEXT] 어머니와 함께 가시기 좋은 창원 가포 지역의 오션뷰 카페들을 찾아보았습니다. 총 10곳의 후보를 확보했으며, 상세 정보는 아래 파일에서 확인하실 수 있습니다.

**[수집 결과]**
*   **검색 지역:** 창원 가포
*   **업종:** 카페
*   **수집 개수:** 10곳
*   **상세 파일:** `./tool_file\1786435145109_collect_places.json`

어머니를 모시고 가시는 만큼, 위 리스트 중 평점이 좋고 뷰가 탁 트인 곳을 골라 방문하시면 좋을 것 같습니다. 특정 분위기(조용한 곳, 디저트가 맛있는 곳 등)를 더 선호하신다면 추가로 말씀해 주세요!
